In [ ]:
# Cell 1: Install required packages
!pip install gradio nest_asyncio



In [ ]:
import gradio as gr
import ast
import os
import base64
from typing import Dict, List, Any
import nest_asyncio

# Apply nest_asyncio to fix event loop issues in Colab
nest_asyncio.apply()

# ============================================
# AST Analyzer Class
# ============================================
class ASTAnalyzer:
    """Handles Python functions with AST-powered analysis"""

    def __init__(self, source_code: str):
        self.source_code = source_code
        try:
            self.tree = ast.parse(source_code)
        except SyntaxError as e:
            self.tree = None
            self.error = str(e)

    def analyze_function(self) -> Dict[str, Any]:
        """Analyze function and extract all metadata"""
        if not self.tree:
            return {}

        for node in ast.walk(self.tree):
            if isinstance(node, ast.FunctionDef):
                return {
                    'name': node.name,
                    'args': [arg.arg for arg in node.args.args],
                    'arg_types': self._get_arg_types(node),
                    'returns': self._get_return_type(node),
                    'docstring': ast.get_docstring(node),
                    'has_recursion': self._check_recursion(node),
                    'has_loops': any(isinstance(stmt, (ast.For, ast.While)) for stmt in ast.walk(node)),
                    'has_conditionals': any(isinstance(stmt, ast.If) for stmt in ast.walk(node))
                }
        return {}

    def _get_arg_types(self, node):
        types = {}
        for arg in node.args.args:
            if arg.annotation:
                if isinstance(arg.annotation, ast.Name):
                    types[arg.arg] = arg.annotation.id
                else:
                    types[arg.arg] = 'Any'
            else:
                types[arg.arg] = 'Any'
        return types

    def _get_return_type(self, node):
        if node.returns:
            if isinstance(node.returns, ast.Name):
                return node.returns.id
        return 'None'

    def _check_recursion(self, node):
        for stmt in ast.walk(node):
            if isinstance(stmt, ast.Call):
                if isinstance(stmt.func, ast.Name) and stmt.func.id == node.name:
                    return True
        return False


# ============================================
# Docstring Generator Class
# ============================================
class DocstringGenerator:
    """Generates professional docstrings for Python functions"""

    STYLES = ['Google', 'NumPy', 'Sphinx', 'reStructuredText']

    def __init__(self, style='Google'):
        self.style = style

    def generate(self, func_info: Dict[str, Any]) -> str:
        """Generate docstring based on selected style"""
        if self.style == 'Google':
            return self._google_style(func_info)
        elif self.style == 'NumPy':
            return self._numpy_style(func_info)
        elif self.style == 'Sphinx':
            return self._sphinx_style(func_info)
        else:
            return self._rest_style(func_info)

    def _google_style(self, func_info):
        lines = []
        if func_info['name'] == 'calculate_factorial':
            lines.append('"""Calculate the factorial of a given number.')
        else:
            lines.append(f'"""Calculate and process data using {func_info["name"]}.')
        lines.append('')

        if func_info['has_recursion']:
            lines.append('This function uses recursion to compute the result.')
        elif func_info['has_loops']:
            lines.append('This function uses loops to process the input data.')
        else:
            lines.append('This function performs operations based on input parameters.')
        lines.append('')

        if func_info['args']:
            lines.append('Args:')
            for arg in func_info['args']:
                arg_type = func_info['arg_types'].get(arg, 'Any')
                if arg == 'n' and func_info['name'] == 'calculate_factorial':
                    lines.append(f'    {arg} ({arg_type}): Integer to compute factorial for')
                else:
                    lines.append(f'    {arg} ({arg_type}): Description of {arg} parameter')
            lines.append('')

        lines.append('Returns:')
        if func_info['name'] == 'calculate_factorial':
            lines.append(f'    {func_info["returns"]}: Factorial of the input number')
        else:
            lines.append(f'    {func_info["returns"]}: Description of return value')
        lines.append('"""')
        return '\n'.join(lines)

    def _numpy_style(self, func_info):
        lines = [f'"""']
        if func_info['name'] == 'calculate_factorial':
            lines.append('Calculate the factorial of a given number')
        else:
            lines.append(func_info['name'])
        lines.append('-' * len(func_info['name']))
        lines.append('')

        if func_info['args']:
            lines.append('Parameters')
            lines.append('----------')
            for arg in func_info['args']:
                arg_type = func_info['arg_types'].get(arg, 'Any')
                lines.append(f'{arg} : {arg_type}')
                if arg == 'n' and func_info['name'] == 'calculate_factorial':
                    lines.append(f'    Input integer to compute factorial')
                else:
                    lines.append(f'    Description of {arg}')
            lines.append('')

        lines.append('Returns')
        lines.append('-------')
        lines.append(f'{func_info["returns"]}')
        if func_info['name'] == 'calculate_factorial':
            lines.append('    Factorial value of input number')
        else:
            lines.append('    Description of return value')
        lines.append('"""')
        return '\n'.join(lines)

    def _sphinx_style(self, func_info):
        lines = [f'"""']
        if func_info['name'] == 'calculate_factorial':
            lines.append('Calculate the factorial of a given number.')
        else:
            lines.append(f'{func_info["name"]} function.')
        lines.append('')

        for arg in func_info['args']:
            arg_type = func_info['arg_types'].get(arg, 'Any')
            if arg == 'n' and func_info['name'] == 'calculate_factorial':
                lines.append(f':param {arg}: Input integer to compute factorial')
            else:
                lines.append(f':param {arg}: Description of {arg}')
            lines.append(f':type {arg}: {arg_type}')

        if func_info['name'] == 'calculate_factorial':
            lines.append(f':returns: Factorial of input number')
        else:
            lines.append(f':returns: Description of return value')
        lines.append(f':rtype: {func_info["returns"]}')
        lines.append('"""')
        return '\n'.join(lines)

    def _rest_style(self, func_info):
        if func_info['name'] == 'calculate_factorial':
            return f'''"""
Calculate the factorial of a given number.

:param n: Input integer to compute factorial
:type n: int
:returns: Factorial of input number
:rtype: {func_info["returns"]}
"""'''
        else:
            return f'''"""
{func_info["name"]}

This function processes input data and returns results.

:param args: Variable length argument list
:type args: tuple
:returns: Processed result
:rtype: {func_info["returns"]}
"""'''


# ============================================
# Processing Functions
# ============================================
def validate_file(file):
    """Validate file type - only .py and .txt allowed"""
    if file is not None:
        file_ext = os.path.splitext(file.name)[1].lower()
        if file_ext not in ['.py', '.txt']:
            return False, f"❌ Error: Invalid file type '{file_ext}'. Only .py and .txt allowed!"
    return True, "✓ File ready"


def process_function(code_input, style_choice, file_input):
    """Process the function and generate docstring"""
    upload_status = "No file uploaded"

    # Handle file upload if present
    if file_input is not None:
        is_valid, message = validate_file(file_input)
        if not is_valid:
            return code_input, message, message, "❌ Error"

        try:
            with open(file_input.name, 'r', encoding='utf-8') as f:
                file_content = f.read()
            if file_content.strip():
                code_input = file_content
                upload_status = f"✓ Uploaded: {os.path.basename(file_input.name)}"
            else:
                return code_input, "❌ Error: File is empty", "❌ Error: Empty file", "❌ Error"
        except Exception as e:
            return code_input, f"❌ Error reading file: {str(e)}", f"❌ Error: {str(e)}", "❌ Error"

    # Check if code is empty
    if not code_input or not code_input.strip():
        return code_input, "❌ Error: No function provided", upload_status, "❌ Error"

    # Analyze code
    analyzer = ASTAnalyzer(code_input)
    func_info = analyzer.analyze_function()

    if not func_info:
        return code_input, "❌ Error: No valid Python function found", upload_status, "❌ Error"

    # Generate docstring
    generator = DocstringGenerator(style_choice)
    docstring = generator.generate(func_info)

    # Insert docstring into function
    lines = code_input.split('\n')
    for i, line in enumerate(lines):
        if line.strip().startswith('def '):
            doc_lines = docstring.split('\n')
            for j, doc_line in enumerate(doc_lines):
                if doc_line.strip():
                    lines.insert(i + 1 + j, '    ' + doc_line)
            break

    modified_code = '\n'.join(lines)
    success_message = f"✅ Generated {style_choice} style docstring"

    return modified_code, success_message, upload_status, "✅ Ready"


def create_share_link(code):
    """Create a shareable link"""
    if code and code.strip():
        encoded = base64.b64encode(code.encode()).decode()
        short_code = encoded[:30] + "..." if len(encoded) > 30 else encoded
        return f"https://doc-genie.dev/share?code={short_code}"
    return "No code to share"


def clear_all():
    """Clear all inputs and outputs"""
    return (
        "def calculate_factorial(n):\n    if n == 0:\n        return 1\n    return n * calculate_factorial(n-1)",
        "",
        "Waiting for input...",
        "No file uploaded",
        "Ready to export"
    )


def handle_file_upload(file, current_code):
    """Handle file upload with validation"""
    if file is not None:
        is_valid, message = validate_file(file)
        if not is_valid:
            return message, current_code
        else:
            try:
                with open(file.name, 'r', encoding='utf-8') as f:
                    content = f.read()
                return f"✓ Uploaded: {os.path.basename(file.name)}", content
            except:
                return "❌ Error reading file", current_code
    return "No file uploaded", current_code


# ============================================
# CSS Styling - Exact UI Match
# ============================================
custom_css = """
/* Main container - Dark theme */
.gradio-container {
    max-width: 1400px !important;
    margin: auto !important;
    background-color: #1e1e1e !important;
    color: #d4d4d4 !important;
    font-family: 'Segoe UI', system-ui, sans-serif !important;
}

/* Header Section */
#title-header {
    background: linear-gradient(135deg, #0f2027, #203a43, #2c5364);
    padding: 30px 35px;
    border-radius: 15px 15px 0 0;
    margin-bottom: 25px;
    border: 1px solid #3a5f6f;
    text-align: center;
}

#title-header h1 {
    font-size: 42px;
    font-weight: 700;
    color: #ffffff;
    margin: 0;
}

#title-header h2 {
    font-size: 20px;
    color: #a8d0e0;
    margin: 12px 0 8px 0;
}

#title-header p {
    color: #c0d0d8;
    font-size: 15px;
}

/* Section Headers */
.section-header {
    background: linear-gradient(145deg, #2d2d2d, #262626);
    padding: 15px 20px;
    border-radius: 8px;
    margin: 20px 0 15px 0;
    border-left: 6px solid #00bcd4;
    color: #00bcd4;
    font-weight: 600;
    font-size: 16px;
}

/* Box Line Separator */
.box-line {
    border-top: 2px solid #333333;
    margin: 20px 0;
    position: relative;
    text-align: center;
}

.box-line::before {
    content: "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━";
    color: #00bcd4;
    font-size: 14px;
    position: absolute;
    top: -10px;
    left: 50%;
    transform: translateX(-50%);
    background-color: #1e1e1e;
    padding: 0 15px;
}

/* Code input */
.code-input textarea {
    background-color: #1a1a1a !important;
    color: #d4d4d4 !important;
    border: 2px solid #333 !important;
    border-radius: 8px !important;
    font-family: 'Courier New', monospace !important;
    font-size: 14px !important;
}

/* Upload Area */
.upload-container {
    border: 2px dashed #00bcd4;
    border-radius: 12px;
    padding: 30px 20px;
    text-align: center;
    background: linear-gradient(145deg, #252525, #202020);
    margin: 15px 0;
}

.upload-container p:first-child {
    color: #00bcd4;
    font-weight: 600;
    font-size: 18px;
}

.upload-container p:last-child {
    font-size: 12px;
    color: #666;
}

/* Status Box */
.status-box {
    background-color: #2a2a2a;
    border: 1px solid #404040;
    border-radius: 6px;
    padding: 12px 15px;
    margin: 10px 0;
    position: relative;
    padding-left: 35px;
}

.status-box::before {
    content: "●";
    color: #00bcd4;
    font-size: 20px;
    position: absolute;
    left: 12px;
    top: 8px;
}

.status-box.success::before {
    color: #4caf50;
}

/* Controls Section */
.controls-section {
    background-color: #252525;
    border: 2px solid #333;
    border-radius: 12px;
    padding: 25px;
    margin: 10px 5px;
}

/* Buttons */
.gr-button-primary {
    background: linear-gradient(145deg, #00bcd4, #008c9e) !important;
    border: none !important;
    color: white !important;
    font-weight: 600 !important;
    padding: 12px 24px !important;
    border-radius: 6px !important;
}

.gr-button-secondary {
    background: #333333 !important;
    border: 1px solid #444 !important;
    color: white !important;
    padding: 12px 24px !important;
    border-radius: 6px !important;
}

/* Output Section */
.output-card {
    background-color: #1a1a1a;
    border: 2px solid #00bcd4;
    border-radius: 12px;
    padding: 20px;
    margin: 10px 5px;
}

/* Share Section */
.share-section {
    background: linear-gradient(145deg, #252525, #1e1e1e);
    border: 2px solid #00bcd4;
    border-radius: 12px;
    padding: 20px;
    margin: 20px 0;
    display: flex;
    align-items: center;
    gap: 20px;
    flex-wrap: wrap;
}

.share-label {
    color: #00bcd4;
    font-weight: 600;
    font-size: 16px;
}

.share-options {
    display: flex;
    gap: 20px;
    color: #b0b0b0;
}

.share-link-box {
    background-color: #0a0a0a;
    border: 2px solid #333;
    border-radius: 8px;
    padding: 10px 15px;
    color: #00bcd4;
    font-family: monospace;
    flex-grow: 1;
}

.share-button {
    background: #00bcd4 !important;
    color: white !important;
    border: none !important;
    padding: 10px 20px !important;
    border-radius: 6px !important;
    cursor: pointer !important;
}

/* Export Status */
.export-status {
    background-color: #263238;
    border: 2px solid #00bcd4;
    border-radius: 8px;
    padding: 12px 20px;
    margin: 15px 0;
    color: #00bcd4;
    font-weight: 600;
    text-align: center;
}

/* Status Indicator */
.status-indicator {
    text-align: right;
    padding: 15px;
    color: #00bcd4;
    border-top: 2px solid #333;
    margin-top: 30px;
}

.status-dot {
    display: inline-block;
    width: 10px;
    height: 10px;
    background-color: #4caf50;
    border-radius: 50%;
    margin-right: 8px;
    box-shadow: 0 0 10px #4caf50;
}

/* Error Message */
.error-text {
    color: #ff4444 !important;
    font-weight: 600;
}

.success-text {
    color: #4caf50 !important;
    font-weight: 600;
}
"""


# ============================================
# Build UI - CSS in gr.Blocks (Correct way for Gradio 4.x)
# ============================================
with gr.Blocks(css=custom_css) as demo:

    # Header
    gr.HTML("""
    <div id="title-header">
        <h1># Doc-Genie v2.0</h1>
        <h2>## Professional Python Function Docstring Generator</h2>
        <p>handles your Python functions with self-powered, accurate documentation</p>
    </div>
    """)

    # Box Line Separator
    gr.HTML('<div class="box-line"></div>')

    # Input Function Code Section
    gr.HTML('<div class="section-header">📥 Input Function Code</div>')

    with gr.Row():
        # Left Column - Input
        with gr.Column(scale=1):
            gr.Markdown("- **Python Function**")
            gr.Markdown("  - Rectangular_Snip")

            code_input = gr.Code(
                label="",
                value="def calculate_factorial(n):\n    if n == 0:\n        return 1\n    return n * calculate_factorial(n-1)",
                language="python",
                lines=8,
                elem_classes="code-input"
            )

            gr.Markdown("#### Or Upload any File")

            # Custom Upload Area
            gr.HTML("""
            <div class="upload-container">
                <p>📁 Drop File Here</p>
                <p>Click to Upload</p>
            </div>
            """)

            file_input = gr.File(
                label="",
                file_types=[".py", ".txt"],
                visible=True
            )

            gr.Markdown("**Upload Status**")
            upload_status = gr.Textbox(
                label="",
                value="No file uploaded",
                interactive=False,
                elem_classes="status-box"
            )

        # Right Column - Controls and Output
        with gr.Column(scale=1):
            gr.HTML('<div class="section-header">🎮 Controls</div>')

            style_dropdown = gr.Dropdown(
                choices=["Google", "NumPy", "Sphinx", "reStructuredText"],
                value="Google",
                label="Docstring Style"
            )

            with gr.Row():
                generate_btn = gr.Button("🚀 Generate", variant="primary", elem_classes="gr-button-primary")
                clear_btn = gr.Button("🗑️ Clear", elem_classes="gr-button-secondary")

            gr.Markdown("- **Export Status**")
            export_status = gr.Textbox(
                label="",
                value="Ready to export",
                interactive=False,
                elem_classes="export-status"
            )

            gr.Markdown("#### Generated Output")
            gr.Markdown("- **Function with Docstring**")

            output_code = gr.Code(
                label="",
                language="python",
                lines=12,
                interactive=False,
                elem_classes="code-output"
            )

            gr.Markdown("**Status**")
            process_status = gr.Textbox(
                label="",
                value="Waiting for input...",
                interactive=False,
                elem_classes="status-box"
            )

    # Export Status Section
    gr.HTML('<div class="section-header">📤 Export Status</div>')

    # Share Section
    with gr.Row():
        share_link_display = gr.Textbox(
            label="Share Link",
            value="Ready to generate link",
            interactive=False,
            elem_classes="share-link-box"
        )
        share_btn = gr.Button("🔗 Generate Link", elem_classes="share-button")

    # Status Indicator
    gr.HTML("""
    <div class="status-indicator">
        <span class="status-dot"></span>
        Status: <strong>🟢 Online - Ready to process</strong>
    </div>
    """)

    # ============================================
    # Event Handlers
    # ============================================

    # Generate button click
    generate_btn.click(
        fn=process_function,
        inputs=[code_input, style_dropdown, file_input],
        outputs=[output_code, process_status, upload_status, export_status]
    )

    # Clear button
    clear_btn.click(
        fn=clear_all,
        outputs=[code_input, output_code, process_status, upload_status, export_status]
    ).then(
        fn=lambda: "Ready to generate link",
        outputs=[share_link_display]
    )

    # File upload handler
    file_input.change(
        fn=handle_file_upload,
        inputs=[file_input, code_input],
        outputs=[upload_status, code_input]
    )

    # Share button
    share_btn.click(
        fn=lambda code: create_share_link(code) if code else "No code to share",
        inputs=[output_code],
        outputs=[share_link_display]
    )

/tmp/ipykernel_389/2675684900.py:559: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as demo:


In [ ]:
# Cell 3: Launch the application
demo.launch(share=True, debug=False)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d914dfb60de2077adf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
